In [1]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import awkward as ak

In [2]:
file = "/home/users/nandan/tmp/phase2_v1/CMSSW_14_2_0_pre1/src/NGTRawSizeImpact/PixelCluster/test/output.root"  # replace with your ROOT file path
print(file)

/home/users/nandan/tmp/phase2_v1/CMSSW_14_2_0_pre1/src/NGTRawSizeImpact/PixelCluster/test/output.root


In [3]:
def data_loader(file_path, var, tree, signal, frac=0):
    print('calling')
    with uproot.open(file)[f'sep19_2_2_dump_raw/{tree}'] as input_tree:
         df = input_tree.arrays(var)
         df = df[(df['target']==signal)]
         df = pd.DataFrame({k: ak.to_list(df[k]) for k in df.fields})
         if frac:
             df = df.sample(frac=frac, random_state=42)
         return df

In [4]:
def plot_var_dist(df, varlists):
   for var in varlists:
       if var == 'target': continue
       print(var)
       minimum_value = min(df[var])
       maximum_value = max(df[var])
       bins = np.linspace(minimum_value, maximum_value, 100)
       for label in [0,1]:
           data = df[(df['target'] == label)][var]
           weights = np.ones_like(data) / len(data)
           plt.hist(data, weights=weights, histtype='step', \
            label='sig' if label==1 else 'bkg', bins=bins)
       plt.legend(loc='best', title=var)
       plt.ylim([0.0001, 10])
       plt.yscale('log')
       plt.savefig(f'{var}.png')
       plt.show()

In [6]:
vars = ['target',
        'size', 'charge', 'x', 'y']            
print('loading sig')
sig = data_loader(file, vars, 'tree', 1, 1)
print('loading bg')
bkg = data_loader(file, vars, 'tree', 0, 1)
df = pd.concat([sig, bkg], ignore_index=True)
df[[f'adc_at_idx_{i}' for i in range(4)]] = pd.DataFrame(df['adcs_ten'].apply(lambda x: x[:4]).tolist(), index=df.index)

loading sig
calling
loading bg
calling


KeyError: 'adcs_ten'

In [ ]:
print(vars)
plot_var_dist(df, vars)
print('plotting done')